# 10 · PHP2Shell example — what changes in a snapshot when two exploits are added

**Question.** If the KEV catalogue grows by two entries, does a ThreatGraph snapshot *show* it, and can we prove *what* changed?

**Idea (route C).** Take the real baseline snapshot (**2025-07-28**, never modified) and build a *second* snapshot = baseline + **two new rows** for the PHP2Shell exploits, as if they had been added to KEV and NVD by today (**2026-10-26**). Then:

1. compare the SHA-256 of every table (which tables differ?);
2. compare the `content_id` (does the whole snapshot differ?);
3. run the diff (what exactly changed?) and check it against a ground truth written **before** the diff;
4. check that *baseline + diff = new snapshot* and that damage to the new snapshot is detected.

**Important — honest scope.** The two rows are a **simulation** (marked as such in the data). We have *one* real upstream version; the second is built by hand. The CVE identifiers and the dates come from press reports and were **not** checked against NVD or CISA KEV. The notebook proves how the *snapshot mechanism* behaves, not facts about PHP2Shell.

No database is touched: everything is done on the snapshot files (baseline is only read; the new snapshot is written to a temporary folder).

## 0 · Setup

In [1]:
import sys, os, io, json, shutil, tempfile
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config" / "sources.yaml").exists())
sys.path[:0] = [str(ROOT / "src")]
import pandas as pd
from IPython.display import display
from threatgraph import config, cleaning as C, jsonld as LD, snapshot as SN
from threatgraph.integrity import sha256_file

pd.set_option("display.max_columns", 30, "display.max_colwidth", 70, "display.width", 200)
SRC = config.load_sources(); LABEL = SRC["common_version"]["label"]          # '2025-07-28'
BASE = config.SNAPSHOT_DIR / LABEL
assert BASE.is_dir(), f"baseline snapshot missing: run `python manage.py snapshot_db` (notebook 06) first -> {BASE}"
TMP = Path(tempfile.mkdtemp(prefix="tg_php2shell_"))
TODAY = "2026-10-26"
checks = []
def check(name, ok, detail=""):
    checks.append({"name": name, "ok": bool(ok), "detail": str(detail)[:200]}); print(("OK  " if ok else "FAIL"), name, detail); return bool(ok)
print("baseline:", BASE); print("temp dir:", TMP)

baseline: /home/claude/threatgraph-ld/data/snapshots/2025-07-28
temp dir: /tmp/tg_php2shell_14uxota_


## 1 · The baseline (version A, 2025-07-28)

We only **read** it. First the fixity check (is it intact?), then the table sizes and hashes.
`snapshot.json` stores a **SHA-256 per table** (one CSV file per table) plus one `content_id` for the whole snapshot.

In [2]:
manA = SN.read_manifest(BASE); A = SN.read_tables(BASE)       # A = 9 frames of strings ('' = NULL)
check("baseline passes verify_snapshot", SN.verify_snapshot(BASE) == [])
display(pd.DataFrame(manA["tables"]).T[["rows", "bytes", "sha256"]].assign(sha256=lambda d: d.sha256.str[:16] + "…"))
print("content_id A:", manA["content_id"])

OK   baseline passes verify_snapshot 


,rows,bytes,sha256
kev_vulnerability,1391,706216,fc0c9410db476d57…
kev_cwe,1328,29558,b84bc6036a202b16…
nvd_cve,1391,724852,435b99c03249cd74…
nvd_cve_cwe,2523,148888,51ec143965c24292…
attack_tactic,14,1285,7e2e881702e4765d…
attack_technique,799,1195845,912ea691422248ae…
attack_technique_tactic,1051,16182,3008af112e5ab1b4…
pkev_mapping,1183,518104,6b76edf9253c3e84…
pkev_mapping_reference,2256,224856,c9111e106ef37a8d…


content_id A: 74d833caed5fc33635e2469d02e09b736a217ccc115460cbf7e203b521105e0a


## 2 · The two new rows (simulated)

Only two tables are involved, because they are the two *sources* we want to simulate:

| Source | Table | New rows |
|---|---|---|
| CISA KEV | `kev_vulnerability` | 2 |
| NVD (fkie-cad mirror) | `nvd_cve` | 2 (the schema says: **no NVD record without a KEV row**, so they go together) |

Everything not known is left **empty (NULL)**, not invented: CVSS scores and vectors are empty (the CVEs would not be scored yet — status `Received`); no CWE, no ATT&CK mapping (`kev_cwe`, `nvd_cve_cwe`, `pkev_mapping` stay untouched).
`due_date` = `date_added` + 21 days (the usual KEV deadline of three weeks; here an assumption). `src_row` continues the numbering of the baseline file.

In [3]:
NEW_IDS = ["CVE-2026-109618", "CVE-2026-109619"]
NOTE = "SIMULATED ENTRY for the course demo - not a CISA KEV record; CVE id and dates not verified"

kev_new = pd.DataFrame([
  {"cve_id": NEW_IDS[0], "vendor_project": "PHP Group", "product": "PHP",
   "vulnerability_name": "PHP2Shell PHP Remote Code Execution Vulnerability (simulated)",
   "date_added": TODAY, "due_date": "2026-11-16",
   "short_description": "PHP contains a flaw that allows a remote attacker to run arbitrary code on the server (first PHP2Shell exploit; simulated entry).",
   "required_action": "Apply updates per vendor instructions.", "ransomware_use": "unknown",
   "notes": NOTE, "due_before_added": "false", "src_row": "1391"},
  {"cve_id": NEW_IDS[1], "vendor_project": "PHP Group", "product": "PHP",
   "vulnerability_name": "PHP2Shell PHP Code Injection Vulnerability (simulated)",
   "date_added": TODAY, "due_date": "2026-11-16",
   "short_description": "PHP contains a code injection flaw that is chained with the first PHP2Shell exploit to obtain a shell (simulated entry).",
   "required_action": "Apply updates per vendor instructions.", "ransomware_use": "unknown",
   "notes": NOTE, "due_before_added": "false", "src_row": "1392"}])

nvd_new = pd.DataFrame([
  {"cve_id": NEW_IDS[0], "source_identifier": "cve@mitre.org", "published": "2026-10-20T10:00:00.000Z",
   "last_modified": "2026-10-20T10:00:00.000Z", "vuln_status": "Received",
   "description_en": "PHP2Shell (simulated record): a remote code execution flaw in PHP. Not a real NVD record.",
   "cvss_v31_score": "", "cvss_v31_severity": "", "cvss_v31_vector": "", "cvss_v31_source": "", "cvss_v31_type": "",
   "cvss_v2_score": "", "n_references": "0"},
  {"cve_id": NEW_IDS[1], "source_identifier": "cve@mitre.org", "published": "2026-10-20T10:00:00.000Z",
   "last_modified": "2026-10-20T10:00:00.000Z", "vuln_status": "Received",
   "description_en": "PHP2Shell (simulated record): a code injection flaw in PHP, used together with the first exploit. Not a real NVD record.",
   "cvss_v31_score": "", "cvss_v31_severity": "", "cvss_v31_vector": "", "cvss_v31_source": "", "cvss_v31_type": "",
   "cvss_v2_score": "", "n_references": "0"}])

# the rows must obey the project schema: same columns, valid ids / enums, new keys, foreign key to KEV
for name, df in (("kev_vulnerability", kev_new), ("nvd_cve", nvd_new)):
    check(f"{name}: columns == schema", list(df.columns) == list(C.SCHEMA[name].columns))
    check(f"{name}: key not in baseline", not set(df.cve_id) & set(A[name].cve_id))
check("cve_id matches the schema pattern", all(C.CVE.match(x) for x in NEW_IDS) if hasattr(C.CVE, "match") else True)
check("ransomware_use in enum", set(kev_new.ransomware_use) <= {"known", "unknown"})
check("vuln_status in enum", set(nvd_new.vuln_status) <= C.VULN_STATUS)
check("FK: every new nvd_cve row has a new kev_vulnerability row", set(nvd_new.cve_id) <= set(kev_new.cve_id))
check("due_date = date_added + 21 days", all((pd.Timestamp(d) - pd.Timestamp(a)).days == 21 for a, d in zip(kev_new.date_added, kev_new.due_date)))
check("src_row continues the baseline numbering", int(kev_new.src_row.astype(int).min()) == int(A["kev_vulnerability"].src_row.astype(int).max()) + 1)

print("\n=== kev_vulnerability: the 2 rows added (all 12 columns) ===")
display(kev_new.T)
print("=== nvd_cve: the 2 rows added (all 13 columns; '' = NULL) ===")
display(nvd_new.T)

OK   kev_vulnerability: columns == schema 
OK   kev_vulnerability: key not in baseline 
OK   nvd_cve: columns == schema 
OK   nvd_cve: key not in baseline 
OK   cve_id matches the schema pattern 
OK   ransomware_use in enum 
OK   vuln_status in enum 
OK   FK: every new nvd_cve row has a new kev_vulnerability row 
OK   due_date = date_added + 21 days 
OK   src_row continues the baseline numbering 

=== kev_vulnerability: the 2 rows added (all 12 columns) ===


,0,1
cve_id,CVE-2026-109618,CVE-2026-109619
vendor_project,PHP Group,PHP Group
product,PHP,PHP
vulnerability_name,PHP2Shell PHP Remote Code Execution Vulnerability (simulated),PHP2Shell PHP Code Injection Vulnerability (simulated)
date_added,2026-10-26,2026-10-26
due_date,2026-11-16,2026-11-16
short_description,PHP contains a flaw that allows a remote attacker to run arbitrary...,PHP contains a code injection flaw that is chained with the first ...
required_action,Apply updates per vendor instructions.,Apply updates per vendor instructions.
ransomware_use,unknown,unknown
notes,SIMULATED ENTRY for the course demo - not a CISA KEV record; CVE i...,SIMULATED ENTRY for the course demo - not a CISA KEV record; CVE i...


=== nvd_cve: the 2 rows added (all 13 columns; '' = NULL) ===


,0,1
cve_id,CVE-2026-109618,CVE-2026-109619
source_identifier,cve@mitre.org,cve@mitre.org
published,2026-10-20T10:00:00.000Z,2026-10-20T10:00:00.000Z
last_modified,2026-10-20T10:00:00.000Z,2026-10-20T10:00:00.000Z
vuln_status,Received,Received
description_en,PHP2Shell (simulated record): a remote code execution flaw in PHP....,"PHP2Shell (simulated record): a code injection flaw in PHP, used t..."
cvss_v31_score,,
cvss_v31_severity,,
cvss_v31_vector,,
cvss_v31_source,,


## 3 · Ground truth — written BEFORE building and comparing

If we computed the expected result from the diff itself, the check would be circular. So we state it now, by hand:

* `kev_vulnerability`: **+2** rows, `nvd_cve`: **+2** rows;
* all other **7 tables**: byte-identical, same SHA-256;
* no row removed, no cell changed;
* `content_id` differs (it is computed from the table hashes), and the new snapshot declares the baseline as its previous version.

In [4]:
EXPECTED = {t: {"added": 0, "removed": 0, "changed": 0} for t in SN.TABLES}
EXPECTED["kev_vulnerability"]["added"] = 2; EXPECTED["nvd_cve"]["added"] = 2
SAME_TABLES = sorted(set(SN.TABLES) - {"kev_vulnerability", "nvd_cve"})
print("tables expected to be identical:", SAME_TABLES)

tables expected to be identical: ['attack_tactic', 'attack_technique', 'attack_technique_tactic', 'kev_cwe', 'nvd_cve_cwe', 'pkev_mapping', 'pkev_mapping_reference']


## 4 · Build snapshot B (2026-10-26)

B = the baseline tables **plus** the 4 rows, written with the same function as every other snapshot (`create_snapshot`, CSV sorted by primary key). The other 7 tables are passed through unchanged. `previous=BASE` records the lineage.

In [5]:
B = {t: df.copy() for t, df in A.items()}
B["kev_vulnerability"] = pd.concat([A["kev_vulnerability"], kev_new], ignore_index=True)
B["nvd_cve"] = pd.concat([A["nvd_cve"], nvd_new], ignore_index=True)

prov = SN.provenance_from(json.loads((config.CLEAN_DIR / "manifest.json").read_text()), SRC)
prov["simulated"] = "baseline + 2 hand-written KEV/NVD rows (PHP2Shell example, notebook 10)"
BDIR = SN.create_snapshot(None, TMP, TODAY, prov, previous=BASE, csv_bytes={t: SN.frame_to_csv(B[t], t) for t in SN.TABLES})
manB = SN.read_manifest(BDIR)
check("snapshot B passes verify_snapshot", SN.verify_snapshot(BDIR) == [])
check("baseline still intact after the whole run (read-only)", SN.verify_snapshot(BASE) == [])
print("snapshot B written to:", BDIR)

OK   snapshot B passes verify_snapshot 
OK   baseline still intact after the whole run (read-only) 
snapshot B written to: /tmp/tg_php2shell_14uxota_/2026-10-26


## 5 · Test 1 — SHA-256 of every table

This is the cheap, first test: one hash per table. **Equal hash ⇒ same bytes ⇒ same table. Different hash ⇒ the table changed** (it does *not* say how).

In [6]:
cmp = pd.DataFrame({
    "rows A": {t: manA["tables"][t]["rows"] for t in SN.TABLES},
    "rows B": {t: manB["tables"][t]["rows"] for t in SN.TABLES},
    "sha256 A": {t: manA["tables"][t]["sha256"][:16] + "…" for t in SN.TABLES},
    "sha256 B": {t: manB["tables"][t]["sha256"][:16] + "…" for t in SN.TABLES},
    "same hash?": {t: manA["tables"][t]["sha256"] == manB["tables"][t]["sha256"] for t in SN.TABLES}})
display(cmp)
differ = sorted(t for t in SN.TABLES if manA["tables"][t]["sha256"] != manB["tables"][t]["sha256"])
check("exactly kev_vulnerability and nvd_cve differ", differ == ["kev_vulnerability", "nvd_cve"], differ)
check("the 7 other tables have the SAME SHA-256", all(manA["tables"][t]["sha256"] == manB["tables"][t]["sha256"] for t in SAME_TABLES))
check("row counts: +2 and +2", (manB["tables"]["kev_vulnerability"]["rows"], manB["tables"]["nvd_cve"]["rows"]) == (1393, 1393))

,rows A,rows B,sha256 A,sha256 B,same hash?
kev_vulnerability,1391,1393,fc0c9410db476d57…,c8d0c507c673ae76…,False
kev_cwe,1328,1328,b84bc6036a202b16…,b84bc6036a202b16…,True
nvd_cve,1391,1393,435b99c03249cd74…,1e56dc4e9825e231…,False
nvd_cve_cwe,2523,2523,51ec143965c24292…,51ec143965c24292…,True
attack_tactic,14,14,7e2e881702e4765d…,7e2e881702e4765d…,True
attack_technique,799,799,912ea691422248ae…,912ea691422248ae…,True
attack_technique_tactic,1051,1051,3008af112e5ab1b4…,3008af112e5ab1b4…,True
pkev_mapping,1183,1183,6b76edf9253c3e84…,6b76edf9253c3e84…,True
pkev_mapping_reference,2256,2256,c9111e106ef37a8d…,c9111e106ef37a8d…,True


OK   exactly kev_vulnerability and nvd_cve differ ['kev_vulnerability', 'nvd_cve']
OK   the 7 other tables have the SAME SHA-256 
OK   row counts: +2 and +2 


True

## 6 · Test 2 — the whole snapshot: `content_id` and lineage

`content_id` = SHA-256 of the list of table hashes. If any table changes, the `content_id` changes: it is the *fingerprint of the whole snapshot*.
It is a **fixity/identity** value, not a digital signature (anyone could rebuild a snapshot and compute it).

In [7]:
print("content_id A:", manA["content_id"]); print("content_id B:", manB["content_id"])
check("content_id differs", manA["content_id"] != manB["content_id"])
check("B.previous_content_id == A.content_id", manB["previous_content_id"] == manA["content_id"])
check("A has no previous version", not manA.get("previous_content_id"))
check("content_id is recomputable from the 9 table hashes", SN.content_id({t: i["sha256"] for t, i in manB["tables"].items()}) == manB["content_id"])
# the description of B in JSON-LD (DCAT/PROV) links the two versions
node = LD.snapshot_node(manB)
check("JSON-LD of B: wasRevisionOf -> IRI of A", node["wasRevisionOf"] == LD.snapshot_iri(manA["content_id"]), node["wasRevisionOf"])
print(json.dumps({k: node[k] for k in ("@id", "title", "version", "wasRevisionOf")}, indent=1))

content_id A: 74d833caed5fc33635e2469d02e09b736a217ccc115460cbf7e203b521105e0a
content_id B: 870c1ec4b0ef044f08ab4b08dd71bf7e41218874ccaf86e28ba8271027990252
OK   content_id differs 
OK   B.previous_content_id == A.content_id 
OK   A has no previous version 
OK   content_id is recomputable from the 9 table hashes 
OK   JSON-LD of B: wasRevisionOf -> IRI of A urn:threatgraph:snapshot:74d833caed5fc336
{
 "@id": "urn:threatgraph:snapshot:870c1ec4b0ef044f",
 "title": "ThreatGraph-LD snapshot 2026-10-26",
 "version": "2026-10-26",
 "wasRevisionOf": "urn:threatgraph:snapshot:74d833caed5fc336"
}


## 7 · Test 3 — the diff: *what* changed

The hash only says "different". The diff compares the two snapshots **by natural key** (`cve_id`), so it can say which rows were added, removed or changed. Compared with the ground truth of section 3.

In [8]:
D = SN.diff_snapshots(BASE, BDIR)
S = D.summary(); display(S)
got = {t: {"added": int(S.loc[t, "added"]), "removed": int(S.loc[t, "removed"]), "changed": int(S.loc[t, "changed rows"])} for t in SN.TABLES}
check("diff == ground truth (every table, every kind)", got == EXPECTED)
check("diff is not 'identical'", not D.identical)
print("\nchangelog by CVE:"); [print(" ", k, v) for k, v in D.by_cve().items()]

,added,removed,changed rows,changed cells
table,,,,
kev_vulnerability,2,0,0,0
kev_cwe,0,0,0,0
nvd_cve,2,0,0,0
nvd_cve_cwe,0,0,0,0
attack_tactic,0,0,0,0
attack_technique,0,0,0,0
attack_technique_tactic,0,0,0,0
pkev_mapping,0,0,0,0
pkev_mapping_reference,0,0,0,0


OK   diff == ground truth (every table, every kind) 
OK   diff is not 'identical' 

changelog by CVE:
  CVE-2026-109618 ['+ kev_vulnerability', '+ nvd_cve']
  CVE-2026-109619 ['+ kev_vulnerability', '+ nvd_cve']


[None, None]

In [9]:
# the rows the diff reports as added are exactly the rows we inserted (all columns, compared as strings)
for t, new in (("kev_vulnerability", kev_new), ("nvd_cve", nvd_new)):
    got_rows = D.tables[t].added.drop(columns=["src_row"], errors="ignore").set_index("cve_id").sort_index()
    exp_rows = new.drop(columns=["src_row"], errors="ignore").set_index("cve_id").sort_index()
    check(f"{t}: added rows == inserted rows", got_rows.equals(exp_rows))

OK   kev_vulnerability: added rows == inserted rows 
OK   nvd_cve: added rows == inserted rows 


## 8 · Independent recount (no diff code involved)

Pure set arithmetic on the primary keys read straight from the two CSV files.

In [10]:
for t in ("kev_vulnerability", "nvd_cve"):
    ka = set(pd.read_csv(BASE / "tables" / f"{t}.csv", dtype=str).cve_id)
    kb = set(pd.read_csv(BDIR / "tables" / f"{t}.csv", dtype=str).cve_id)
    check(f"{t}: B - A == the 2 PHP2Shell ids, A - B empty", kb - ka == set(NEW_IDS) and not (ka - kb))

OK   kev_vulnerability: B - A == the 2 PHP2Shell ids, A - B empty 
OK   nvd_cve: B - A == the 2 PHP2Shell ids, A - B empty 


## 9 · Patch property: baseline + diff = B, and the inverse

If the diff were wrong or incomplete, applying it to A would not give B.

In [11]:
patched = SN.apply_diff(A, D)
Bn = SN.normalize(SN.read_tables(BDIR))
check("A + diff == B (all 9 tables)", all(patched[t].equals(Bn[t]) for t in SN.TABLES))
D_inv = SN.diff_snapshots(BDIR, BASE)
check("inverse diff: the 4 rows appear as REMOVED", int(D_inv.summary()["removed"].sum()) == 4 and int(D_inv.summary()["added"].sum()) == 0)
check("a snapshot against itself: identical", SN.diff_snapshots(BASE, BASE).identical)

OK   A + diff == B (all 9 tables) 


OK   inverse diff: the 4 rows appear as REMOVED 


OK   a snapshot against itself: identical 


True

## 10 · Negative controls — would the checks notice damage or a wrong claim?

A test that cannot fail proves nothing. We damage a **copy** of B in two ways and expect `verify_snapshot` to complain.

In [12]:
COPY = TMP / "damaged"; shutil.copytree(BDIR, COPY, dirs_exist_ok=True); SN.make_writable(COPY)
f = COPY / "tables" / "nvd_cve.csv"
f.write_text(f.read_text().replace("Received", "Analyzed", 1))                       # one word changed
problems = SN.verify_snapshot(COPY); print(problems)
check("one changed word in nvd_cve.csv is detected", any("nvd_cve" in p for p in problems))

COPY2 = TMP / "damaged2"; shutil.copytree(BDIR, COPY2, dirs_exist_ok=True); SN.make_writable(COPY2)
f2 = COPY2 / "tables" / "kev_vulnerability.csv"; lines = f2.read_text().split("\n")
f2.write_text("\n".join(l for l in lines if NEW_IDS[1] not in l))                    # silently drop one of the new rows
problems2 = SN.verify_snapshot(COPY2); print(problems2)
check("a silently deleted row is detected", any("kev_vulnerability" in p for p in problems2))

# the diff refuses to compare a damaged snapshot
try: SN.diff_snapshots(BASE, COPY); refused = False
except SN.SnapshotError: refused = True
check("diff refuses a damaged snapshot", refused)

# rebuilding B from the same inputs gives the SAME content_id (determinism)
REB = SN.create_snapshot(None, TMP, "rebuild", prov, previous=BASE, csv_bytes={t: SN.frame_to_csv(B[t], t) for t in SN.TABLES})
check("same inputs -> same content_id", SN.read_manifest(REB)["content_id"] == manB["content_id"])

['nvd_cve: size 725230 != manifest 725259', 'nvd_cve: SHA-256 differs from manifest']
OK   one changed word in nvd_cve.csv is detected 
['kev_vulnerability: size 706609 != manifest 706986', 'kev_vulnerability: SHA-256 differs from manifest']
OK   a silently deleted row is detected 


OK   diff refuses a damaged snapshot 


OK   same inputs -> same content_id 


True

## 11 · Summary and gate

In [13]:
res = pd.DataFrame(checks); display(res.assign(detail=res.detail.str[:70]))
print(f"{res.ok.sum()}/{len(res)} checks passed")
assert res.ok.all(), res[~res.ok]

,name,ok,detail
0,baseline passes verify_snapshot,True,
1,kev_vulnerability: columns == schema,True,
2,kev_vulnerability: key not in baseline,True,
3,nvd_cve: columns == schema,True,
4,nvd_cve: key not in baseline,True,
5,cve_id matches the schema pattern,True,
6,ransomware_use in enum,True,
7,vuln_status in enum,True,
8,FK: every new nvd_cve row has a new kev_vulnerability row,True,
9,due_date = date_added + 21 days,True,


34/34 checks passed


## 12 · What this shows — and what it does not

**Shows**
* A change in two source tables changes **exactly those two table hashes**; the other seven stay identical → the hash localises *where* the data changed.
* The `content_id` changes, and the new snapshot points to the old one (`previous_content_id`, `wasRevisionOf`) → versions are chained, the old one is never overwritten.
* SHA-256 answers *"different?"*; the **diff** answers *"what exactly?"* (+2 rows in 2 tables, the CVE ids, every cell) and is validated by a ground truth, an independent recount and the patch property.
* Damaged copies are detected (fixity), so a snapshot can be cited and checked later.

**Does not show**
* That PHP2Shell is in KEV or NVD today: the rows are hand-written, with IDs/dates from press reports, unverified.
* That our *pipeline* ingests a real new release: B is built from A plus 4 rows, not by re-running the download and cleaning of a newer upstream version (that would be the real second snapshot).
* Authenticity: `content_id` is not a signature.

Notebook 06 remains the full proof (44 checks, synthetic changes of all kinds: added, removed, changed, traps on positional columns). This notebook is the readable, up-to-date example of the same mechanism.